# 03 — Stratified 5-fold cross-validation

**Protocol** (`src/data/make_folds.py`, folds versioned in `data/splits/cv/`):
- CV pool = train + val images (1,756), split into 5 stratified folds (seed 42);
- for fold *k*: the model is trained on the other folds (15 % of them kept for early stopping)
  and evaluated on fold *k* → every pool image is evaluated exactly once;
- each fold model is also scored on the **fixed test set** (310 images, never used in the folds);
- same training conditions as the main experiments (`configs/base.yaml`).

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Nothing to edit.

- Runs are saved on Drive (`MyDrive/road-damage-tl/runs_cv/`). If the session stops, just
  `Run all` again: finished folds are skipped.
- Several accounts can run this notebook **at the same time**: a run being trained elsewhere is
  locked and skipped (lock files in `runs_cv/locks/`).
- About 10–15 min per fold on a T4 → about 2.5–3.5 h for the 3 default configurations.

In [ ]:
# Configurations evaluated by cross-validation: the baseline + the best strategy of the main experiments
CV_CONFIGS = [
    ("scratch_cnn", "from_scratch"),     # baseline
    ("alexnet", "partial_finetune"),     # best AlexNet strategy (mean of 3 runs), lightest model
    ("vit_b16", "full_finetune"),        # best macro F1 on the fixed test set
    # optional, if GPU time allows:
    # ("convnextv2", "full_finetune"),   # best AUC
    # ("nfnet_f1", "full_finetune"),     # heaviest model (batch 16 x 2, set automatically)
]
FOLDS = 5
EPOCHS = 30

REPO_URL = "https://github.com/Brahim-semlali/road-damage-tl.git"
DRIVE_ROOT = "/content/drive/MyDrive/road-damage-tl"
REPO_DIR = "/content/road-damage-tl"
RAW_DIR = "/content/data/raw"
CV_DIR = f"{DRIVE_ROOT}/runs_cv"

## 1. Setup: Drive, code, dependencies, data

In [ ]:
import glob, json, os, subprocess, time
from google.colab import drive
drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log --oneline -1
!pip install -q timm albumentations mlflow

if not os.path.isdir(f"{RAW_DIR}/sih_road_dataset"):
    zips = sorted(glob.glob(f"{DRIVE_ROOT}/datasets/*.zip"))
    assert zips, ("No dataset zip in " + DRIVE_ROOT + "/datasets. If the folder road-damage-tl was SHARED with you: "
                  "drive.google.com > Shared with me > right-click road-damage-tl > Organize > Add shortcut > My Drive, "
                  "then Runtime > Restart session and Run all.")
    os.makedirs(RAW_DIR, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", zips[0], "-d", RAW_DIR], check=True)
print(sum(len(f) for _, _, f in os.walk(RAW_DIR)), "files in", RAW_DIR)
assert os.path.exists("data/splits/cv/fold0/train.csv"), "CV folds missing in the repository"

import torch
assert torch.cuda.is_available(), (
    "NO GPU! Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Progress

In [ ]:
LOCK_DIR = f"{CV_DIR}/locks"
LOCK_HOURS = 3          # a lock older than this is considered abandoned (session stopped)
os.makedirs(LOCK_DIR, exist_ok=True)

def is_done(model, strategy, fold):
    for f in glob.glob(f"{CV_DIR}/fold{fold}/{model}__{strategy}__*/metrics.json"):
        if json.load(open(f)).get("device", "cpu") != "cpu":
            return True
    return False

def lock_path(model, strategy, fold):
    return f"{LOCK_DIR}/{model}__{strategy}__fold{fold}.lock"

def is_locked(model, strategy, fold):
    p = lock_path(model, strategy, fold)
    return os.path.exists(p) and time.time() - os.path.getmtime(p) < LOCK_HOURS * 3600

def show_progress():
    for model, strategy in CV_CONFIGS:
        marks = ["done" if is_done(model, strategy, k) else ("running" if is_locked(model, strategy, k) else "-")
                 for k in range(FOLDS)]
        print(f"{model:12s} {strategy:18s} " + "  ".join(f"fold{k}:{m:7s}" for k, m in enumerate(marks)))

show_progress()

## 3. Quick sanity check (2 batches, 1 epoch)

In [ ]:
model, strategy = CV_CONFIGS[-1]
!python -m src.train --model {model} --strategy {strategy} --epochs 1 --max-batches 2 --splits-dir data/splits/cv/fold0 --extra-test data/splits/test.csv --data-root {RAW_DIR} --out /content/debug_runs 2>&1 | grep -E "Run:|Parameters|Epoch|extra_test_f1|Traceback|Error"

## 4. Training (all pending folds, one after the other)

Before each run the list of pending folds is read again from Drive, so this cell cooperates with
other accounts running the same notebook.

In [ ]:
import shutil
DB_LOCAL, DB_DRIVE = "/content/mlflow.db", f"{DRIVE_ROOT}/mlflow.db"
if os.path.exists(DB_DRIVE):
    shutil.copy(DB_DRIVE, DB_LOCAL)

def next_job():
    for model, strategy in CV_CONFIGS:
        for k in range(FOLDS):
            if not is_done(model, strategy, k) and not is_locked(model, strategy, k):
                return model, strategy, k
    return None

while (job := next_job()) is not None:
    model, strategy, k = job
    open(lock_path(model, strategy, k), "w").write(time.ctime())
    bs, acc = (16, 2) if model == "nfnet_f1" else (32, 1)
    print(f"{'=' * 25} {model} / {strategy} / fold {k} {'=' * 25}")
    !python -m src.train --model {model} --strategy {strategy} --epochs {EPOCHS} --batch-size {bs} --accum-steps {acc} --splits-dir data/splits/cv/fold{k} --extra-test data/splits/test.csv --tag cv_fold{k} --data-root {RAW_DIR} --out "{CV_DIR}/fold{k}" --mlflow-uri "sqlite:///{DB_LOCAL}" --mlflow-artifacts "{DRIVE_ROOT}/mlartifacts" 2>&1 | grep -vE "UserWarning|scheduler.step|HF_TOKEN"
    if os.path.exists(lock_path(model, strategy, k)):
        os.remove(lock_path(model, strategy, k))
    if os.path.exists(DB_LOCAL):
        shutil.copy(DB_LOCAL, DB_DRIVE)
    if not is_done(model, strategy, k):
        print("!!! This run did not finish: read the error above. Stopping.")
        break

print()
show_progress()

## 5. Cross-validation results (can be run at any time, even when not all folds are finished)

In [ ]:
!python -W ignore -m src.cv_summary --runs "{CV_DIR}" --out results/cv --folds {FOLDS}

from IPython.display import Image, Markdown, display
display(Markdown(open("results/cv/cv_results.md").read()))
display(Markdown(open("results/cv/cv_pairwise.md").read()))
display(Image("results/cv/cv_f1_per_fold.png", width=800))

## 6. Save (Drive + zip to commit in the repository)

In [ ]:
!mkdir -p "{DRIVE_ROOT}/results/cv" && cp -r results/cv/* "{DRIVE_ROOT}/results/cv/"
!cd /content/road-damage-tl && zip -q -r /content/results_cv.zip results/cv

from google.colab import files
files.download("/content/results_cv.zip")